# RoadSense Exploratory Data Analysis

**Understanding the input data before scoring.**

This notebook examines the raw ADB GPS probe data for Thailand and Maharashtra. It covers:
- Data quality and completeness
- Road network composition (road class, land use distributions)
- Speed characteristics (posted limits, operating speeds)
- Regional comparisons
- Sample size and reliability

Understanding these distributions is essential for interpreting scoring results and identifying potential biases.

In [ ]:
from __future__ import annotations

import warnings
warnings.filterwarnings("ignore")

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from pathlib import Path

import roadsense
_ROADSENSE_ROOT = Path(roadsense.__file__).resolve().parent.parent.parent

from roadsense.data.loader import load_and_clean
from roadsense.config import RAW_DATA_DIR

sns.set_theme(style="whitegrid", palette="muted", font_scale=0.9)
plt.rcParams["figure.dpi"] = 120

---
## 1. Data Ingestion

Load raw ADB datasets through the standard cleaning pipeline. The `load_and_clean()` function applies filters, imputations, and normalisations documented in the methodology.

In [ ]:
df = load_and_clean(RAW_DATA_DIR)

print(f"Total segments after cleaning: {len(df)}")
print(f"Columns: {list(df.columns)}")
print(f"Memory: {df.memory_usage(deep=True).sum() / 1024:.1f} KB")

---
## 2. Data Completeness

Check for missing values across all columns. The cleaning pipeline should have handled these, but it is good practice to verify.

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
completeness = pd.DataFrame({"Missing": missing, "% Missing": missing_pct})
completeness = completeness[completeness["Missing"] > 0]

if len(completeness) == 0:
    print("No missing values in any column. Data is fully complete.")
else:
    print("Columns with missing values:")
    print(completeness.to_string())

---
## 3. Road Network Composition

### 3.1 By Region

The dataset covers two pilot regions: Thailand and Maharashtra, India.

In [ ]:
region_counts = df["region"].value_counts()
print("Segments by region:")
print(region_counts.to_string())

fig, ax = plt.subplots(figsize=(4, 3))
bars = ax.bar(region_counts.index, region_counts.values,
              color=["#3B8BD4", "#EF9F27"], edgecolor="white", linewidth=0.5)
ax.set_ylabel("Segments")
ax.set_title("Segments by Region", fontsize=12, fontweight="semibold")
for bar, val in zip(bars, region_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            str(val), ha="center", va="bottom", fontsize=10)
plt.tight_layout()
plt.show()

### 3.2 By Road Class

The ADB data classifies roads into four functional classes.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))

rc_order = ["motorway", "trunk", "primary", "secondary"]
rc_counts = df["RoadClass"].value_counts()
rc_present = [r for r in rc_order if r in rc_counts.index]
colors_rc = ["#1D9E75", "#3B8BD4", "#EF9F27", "#d32f2f"]

bars = ax.bar(rc_present, [rc_counts[r] for r in rc_present],
              color=colors_rc[:len(rc_present)], edgecolor="white", linewidth=0.5)
ax.set_ylabel("Segments")
ax.set_title("Segments by Road Class", fontsize=12, fontweight="semibold")
for bar, r in zip(bars, rc_present):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            str(rc_counts[r]), ha="center", va="bottom", fontsize=10)
plt.tight_layout()
plt.show()

print("Road class by region:")
print(pd.crosstab(df["region"], df["RoadClass"]).to_string())

### 3.3 By Land Use

Roads are classified as Urban or Rural.

In [ ]:
lu_counts = df["LandUse"].value_counts()
print("Land use distribution:")
print(lu_counts.to_string())

fig, ax = plt.subplots(figsize=(4, 3))
bars = ax.bar(lu_counts.index, lu_counts.values,
              color=["#3B8BD4", "#EF9F27"], edgecolor="white", linewidth=0.5)
ax.set_ylabel("Segments")
ax.set_title("Segments by Land Use", fontsize=12, fontweight="semibold")
for bar, val in zip(bars, lu_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            str(val), ha="center", va="bottom", fontsize=10)
plt.tight_layout()
plt.show()

print("Land use by region:")
print(pd.crosstab(df["region"], df["LandUse"]).to_string())

---
## 4. Speed Characteristics

### 4.1 Posted Speed Limits

Distribution of posted speed limits across the network.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
ax = axes[0]
ax.hist(df["SpeedLimit"], bins=range(20, 130, 10), edgecolor="white",
        color="#3B8BD4", linewidth=0.5, alpha=0.8)
ax.set_xlabel("Posted Speed Limit (km/h)", fontsize=10)
ax.set_ylabel("Segments", fontsize=10)
ax.set_title("Distribution of Posted Speed Limits", fontsize=11, fontweight="semibold")

# By road class
ax = axes[1]
for rc, color in zip(rc_present, colors_rc):
    data = df[df["RoadClass"] == rc]["SpeedLimit"]
    if len(data) == 0:
        continue
    ax.hist(data, bins=range(20, 130, 10), alpha=0.5, label=rc, color=color, edgecolor="white", linewidth=0.3)
ax.set_xlabel("Posted Speed Limit (km/h)", fontsize=10)
ax.set_ylabel("Segments", fontsize=10)
ax.set_title("Speed Limits by Road Class", fontsize=11, fontweight="semibold")
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

print("Speed limit statistics (km/h):")
print(df["SpeedLimit"].describe().round(1).to_string())

### 4.2 Operating Speeds (85th Percentile)

The 85th percentile operating speed represents the speed at or below which 85% of free-flowing traffic travels. It is the standard engineering metric for setting speed limits and evaluating alignment.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.hist(df["F85thPercentileSpeed"], bins=range(10, 150, 10), edgecolor="white",
        color="#d32f2f", linewidth=0.5, alpha=0.8)
ax.set_xlabel("85th Percentile Speed (km/h)", fontsize=10)
ax.set_ylabel("Segments", fontsize=10)
ax.set_title("Distribution of Operating Speeds", fontsize=11, fontweight="semibold")

ax = axes[1]
ax.scatter(df["SpeedLimit"], df["F85thPercentileSpeed"], alpha=0.5, s=15,
           color="#333", edgecolors="white", linewidth=0.3)
lims = [0, 150]
ax.plot(lims, lims, "--", color="#666", linewidth=0.8, label="y = x")
ax.set_xlabel("Posted Speed Limit (km/h)", fontsize=10)
ax.set_ylabel("85th Percentile Speed (km/h)", fontsize=10)
ax.set_title("Posted Limit vs Operating Speed", fontsize=11, fontweight="semibold")
ax.legend(fontsize=8)
ax.set_xlim(0, 150)
ax.set_ylim(0, 150)

plt.tight_layout()
plt.show()

print("Operating speed statistics (km/h):")
print(df["F85thPercentileSpeed"].describe().round(1).to_string())

above_limit = (df["F85thPercentileSpeed"] > df["SpeedLimit"]).mean() * 100
print(f"\nSegments where operating speed exceeds posted limit: {above_limit:.1f}%")

### 4.3 Speed Gap (Operating Speed - Posted Limit)

A positive gap means traffic is moving faster than the posted limit. A large gap suggests either enforcement issues or a limit that is too low for the road design. A negative gap (traffic slower than limit) suggests the limit is higher than drivers consider safe.

In [ ]:
df["speed_gap"] = df["F85thPercentileSpeed"] - df["SpeedLimit"]

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(df["speed_gap"], bins=25, edgecolor="white",
        color="#3B8BD4", linewidth=0.5, alpha=0.8)
ax.axvline(0, color="#d32f2f", linestyle="--", linewidth=0.8, label="F85 = Posted Limit")
ax.set_xlabel("Speed Gap: F85 - Posted Limit (km/h)", fontsize=10)
ax.set_ylabel("Segments", fontsize=10)
ax.set_title("Distribution of Speed Gaps", fontsize=12, fontweight="semibold")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print("Speed gap statistics (km/h):")
print(df["speed_gap"].describe().round(1).to_string())

---
## 5. Sample Size and Reliability

The reliability of operating speed estimates depends on the number of observations per segment. The cleaning pipeline drops segments with fewer than 500 samples.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.hist(df["SampleSize_avg"], bins=30, edgecolor="white",
        color="#1D9E75", linewidth=0.5, alpha=0.8)
ax.axvline(500, color="#d32f2f", linestyle="--", linewidth=0.8, label="Min threshold (500)")
ax.set_xlabel("Average Sample Size", fontsize=10)
ax.set_ylabel("Segments", fontsize=10)
ax.set_title("Sample Size Distribution", fontsize=11, fontweight="semibold")
ax.legend(fontsize=8)

ax = axes[1]
ax.scatter(df["SampleSize_avg"], df["F85thPercentileSpeed"], alpha=0.4, s=10,
           color="#333", edgecolors="white", linewidth=0.3)
ax.set_xlabel("Sample Size", fontsize=10)
ax.set_ylabel("85th Percentile Speed (km/h)", fontsize=10)
ax.set_title("Sample Size vs Operating Speed", fontsize=11, fontweight="semibold")
ax.set_xscale("log")

plt.tight_layout()
plt.show()

print("Sample size statistics:")
print(df["SampleSize_avg"].describe().round(0).to_string())

tiers = [
    (5000, "High", "#1b5e20"),
    (1000, "Medium", "#f57f17"),
]
print("\nReliability tier breakdown:")
for threshold, label, _ in tiers:
    n = (df["SampleSize_avg"] >= threshold).sum()
    print(f"  {label} (>= {threshold}): {n} segments ({n/len(df)*100:.1f}%)")
n_low = (df["SampleSize_avg"] < 1000).sum()
print(f"  Low (< 1000): {n_low} segments ({n_low/len(df)*100:.1f}%)")

---
## 6. Traffic Volume

The WeightedSample field provides an estimate of traffic volume per segment.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(df["WeightedSample"], bins=30, edgecolor="white",
        color="#EF9F27", linewidth=0.5, alpha=0.8)
ax.set_xlabel("Traffic Volume (Weighted Sample)", fontsize=10)
ax.set_ylabel("Segments", fontsize=10)
ax.set_title("Traffic Volume Distribution", fontsize=12, fontweight="semibold")
plt.tight_layout()
plt.show()

print("Traffic volume statistics:")
print(df["WeightedSample"].describe().round(0).to_string())
print(f"\nSkewness: {df['WeightedSample'].skew():.2f}")
print(f"Note: High skew suggests log-transformation is appropriate, which is applied in the pipeline.")

---
## 7. Segment Length

Segment lengths vary. Very short segments may have unreliable speed estimates, while very long segments may mask internal variation.

In [ ]:
df["length_km"] = df["Shape_Length"] / 1000

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(df["length_km"], bins=30, edgecolor="white",
        color="#3B8BD4", linewidth=0.5, alpha=0.8)
ax.set_xlabel("Segment Length (km)", fontsize=10)
ax.set_ylabel("Segments", fontsize=10)
ax.set_title("Segment Length Distribution", fontsize=12, fontweight="semibold")
plt.tight_layout()
plt.show()

print("Segment length statistics (km):")
print(df["length_km"].describe().round(3).to_string())

---
## 8. Key Findings from EDA

1. **Data quality is high** -- no missing values remain after cleaning. All 100 segments pass validation.
2. **Balanced regional split** -- 50 segments each from Thailand and Maharashtra.
3. **Road class diversity** -- all four functional classes are represented, with trunk and primary roads being most common.
4. **Speed limit range** -- 30 to 110 km/h across the network, with rural roads generally having higher limits.
5. **Operating speeds frequently exceed posted limits** -- a significant fraction of the network shows positive speed gaps.
6. **Sample sizes are adequate** -- all segments exceed the 500-observation threshold, with most in the Medium to High reliability tiers.
7. **Traffic volume is right-skewed** -- a small number of high-volume corridors carry a disproportionate share of traffic.

---
*Notebook generated by RoadSense. Data from 100-segment synthetic demo. Full ADB dataset results available under NDA.*